# 🔍 Python Graphs DFS — The Master Guide
### *From Zero to Interview-Ready*

---

> **Mental Model First:**
> DFS is like exploring a cave system with a single torch. You pick one tunnel
> and walk as far as you can go — marking every chamber you pass through.
> When you hit a dead end, you backtrack to the last junction and try the next tunnel.
> You never visit a chamber twice. When every tunnel from every junction is exhausted, you're done.

---

## 📋 Table of Contents

| # | Section |
|---|----------|
| 1 | [What Is Graph DFS? The Visual Model](#1) |
| 2 | [Creating / Setup](#2) |
| 3 | [The Core API — All Operations](#3) |
| 4 | [Decision Map — When To Use What](#4) |
| 5 | [Pattern 1: Number of Islands (LC 200)](#5) |
| 6 | [Pattern 2: Clone Graph (LC 133)](#6) |
| 7 | [Pattern 3: Pacific Atlantic Water Flow (LC 417)](#7) |
| 8 | [Pattern 4: Surrounded Regions (LC 130)](#8) |
| 9 | [Pattern 5: Max Area of Island (LC 695)](#9) |
| 10 | [The Graphs DFS Decision Map](#10) |
| 11 | [Interview Cheat Sheet](#11) |

<a id='1'></a>
## 1. What Is Graph DFS? The Visual Model

```
               GRAPH DFS — THE CAVE EXPLORER

  Graph (adjacency list):
    0 → [1, 2]
    1 → [0, 3]
    2 → [0, 4]
    3 → [1]
    4 → [2]

  DFS from node 0 (recursive, visited = {}):

    visit(0): mark 0 → explore 1
      visit(1): mark 1 → explore 3
        visit(3): mark 3 → explore 1 (already visited, skip)
        ← backtrack to 1
      ← backtrack to 0 → explore 2
      visit(2): mark 2 → explore 4
        visit(4): mark 4 → explore 2 (already visited, skip)
        ← backtrack to 2
      ← backtrack to 0 → done

  Order visited: 0 → 1 → 3 → 2 → 4

  ON A GRID:
  DIRS = [(0,1),(0,-1),(1,0),(-1,0)]  # 4-directional neighbors

    ┌───┬───┬───┐
    │ 1 │ 1 │ 0 │  DFS from (0,0):
    ├───┼───┼───┤  (0,0)→(0,1)→(1,1)→(1,0)→  sink
    │ 1 │ 1 │ 0 │  marks the entire island
    ├───┼───┼───┤
    │ 0 │ 0 │ 1 │
    └───┴───┴───┘

  KEY RULE: Mark visited BEFORE recursing (not after). Otherwise you revisit.
```

<a id='2'></a>
## 2. Creating / Setup

In [ ]:
from collections import defaultdict

# 4-directional moves on a grid (up, down, left, right)
DIRS = [(0, 1), (0, -1), (1, 0), (-1, 0)]

# 8-directional moves (includes diagonals)
DIRS8 = [(0,1),(0,-1),(1,0),(-1,0),(1,1),(1,-1),(-1,1),(-1,-1)]

# adjacency list graph — undirected
graph_undirected = defaultdict(list)
for u, v in [(0,1),(0,2),(1,3),(2,4)]:
    graph_undirected[u].append(v)    # add edge in both directions
    graph_undirected[v].append(u)
print("undirected graph:", dict(graph_undirected))

# adjacency list graph — directed
graph_directed = defaultdict(list)
for u, v in [(0,1),(0,2),(1,3),(2,3),(3,4)]:
    graph_directed[u].append(v)      # one-way edge only
print("directed graph:", dict(graph_directed))

# grid as 2D list — common in island/flood-fill problems
grid = [
    ['1','1','0'],
    ['1','1','0'],
    ['0','0','1']
]
ROWS, COLS = len(grid), len(grid[0])
print(f"grid: {ROWS}x{COLS}")

# is_valid — boundary check for grid DFS
def is_valid(r, c, rows, cols):
    return 0 <= r < rows and 0 <= c < cols

print("graph and grid setup complete.")

<a id='3'></a>
## 3. The Core API — All Operations

```
DFS PATTERNS                COMPLEXITY          WHAT IT DOES
───────────────────────────────────────────────────────────────
Recursive DFS               O(V+E)              explore all reachable nodes
Iterative DFS (explicit stack) O(V+E)           same, without call stack overflow risk
Grid DFS (flood fill)       O(R*C)              mark all cells in a connected region
DFS + visited set            O(V+E)             avoid revisiting in cyclic graphs
DFS + in-progress set        O(V+E)             detect cycles in directed graphs
DFS collecting return values O(V+E)             aggregate subtree info on the way back

THINGS YOU DO NOT DO:
❌  Start DFS without a visited set in a graph with cycles — infinite loop
❌  Mark visited AFTER recursing — you'll enqueue neighbors multiple times
❌  Use DFS for shortest path (unweighted) — use BFS instead
❌  Forget to restore state in backtracking DFS (modify→recurse→restore)
❌  Stack overflow on huge graphs — iterative DFS or BFS preferred
```

In [ ]:
# RECURSIVE DFS — core template
def dfs_recursive(graph, start):
    visited = set()
    order = []

    def dfs(node):
        visited.add(node)       # mark before recursing to block revisits
        order.append(node)
        for neighbor in graph[node]:
            if neighbor not in visited:
                dfs(neighbor)   # dive deeper into the cave

    dfs(start)
    return order

# ITERATIVE DFS — uses explicit stack, avoids recursion depth limit
def dfs_iterative(graph, start):
    visited = set()
    stack = [start]
    order = []
    while stack:
        node = stack.pop()           # LIFO — last pushed = first explored
        if node in visited:
            continue                 # already explored this chamber
        visited.add(node)
        order.append(node)
        for neighbor in graph[node]:
            if neighbor not in visited:
                stack.append(neighbor)  # push for future exploration
    return order

# Demo both on same graph
print("recursive DFS from 0:", dfs_recursive(graph_undirected, 0))
print("iterative DFS from 0:", dfs_iterative(graph_undirected, 0))
print("DFS templates demonstrated.")

<a id='4'></a>
## 4. Decision Map — When To Use What

```
SIGNAL IN THE PROBLEM                   WHAT TO DO
─────────────────────────────────────────────────────────────────
"connected components" (count regions)  DFS/BFS outer loop over all nodes
"flood fill" or "sink the island"       DFS from each unvisited source cell
"can we reach from X to Y"              DFS/BFS + visited set
"all paths from source to dest"         DFS + backtracking (restore state)
"cycle detection in directed graph"     DFS + in-progress (gray) set
"clone graph / deep copy"               DFS + hash map (old → new node)
"reachable from boundary"               DFS from boundary cells inward
"max area / count of region"            DFS returns accumulated size
"shortest path"                         BFS (not DFS)
"topological order"                     DFS post-order or Kahn's algorithm
```

<a id='5'></a>
## 5. 🧩 Pattern 1: Number of Islands — LC 200

---

```
PROBLEM:
  Given a 2D grid of '1' (land) and '0' (water), count the number of islands.
  An island is a group of '1's connected 4-directionally.

TRICK:
  Outer loop over every cell. When you find an unvisited '1', increment counter
  and DFS to sink the entire island (mark all connected land as visited).
  Each DFS call from the outer loop = one complete island.

SLOW MOTION TRACE on grid:
  [['1','1','0'],
   ['1','0','0'],
   ['0','0','1']]

  (0,0)='1', unvisited → count=1, DFS:
    sink (0,0), (0,1), (1,0)   ← connected component 1
  (0,2)='0' skip
  (1,1)='0' skip
  ...
  (2,2)='1', unvisited → count=2, DFS:
    sink (2,2)                  ← isolated component 2
  → answer = 2

KEY INSIGHT:
  Modifying the grid in-place (set '1'→'0') avoids needing a separate visited set.
  Each outer-loop DFS launch = one new island.

TIME:  O(R*C) — every cell visited at most once
SPACE: O(R*C) — recursion stack in worst case (all land)
```

In [ ]:
import copy

def num_islands(grid):
    """
    LC 200 — Number of Islands
    Approach: DFS from each unvisited land cell; sink the island by overwriting '1'→'0'.
    Args:
        grid (List[List[str]]): 2D grid of '1' (land) and '0' (water).
    Returns:
        int: number of distinct islands.
    Time:  O(R*C) — each cell visited at most once total across all DFS calls
    Space: O(R*C) — recursion stack in worst case (solid land grid)
    """
    if not grid:
        return 0
    rows, cols = len(grid), len(grid[0])
    count = 0

    def sink(r, c):
        if r < 0 or r >= rows or c < 0 or c >= cols or grid[r][c] != '1':
            return               # out of bounds or water or already sunk
        grid[r][c] = '0'         # mark this land as visited by sinking it
        sink(r+1, c)             # explore all 4 neighbors to find the full island
        sink(r-1, c)
        sink(r, c+1)
        sink(r, c-1)

    for r in range(rows):
        for c in range(cols):
            if grid[r][c] == '1':  # unvisited land = new island
                count += 1
                sink(r, c)         # erase the entire island before moving on
    return count

# Slow motion on grid1:
# (0,0)=1: count=1, sink(0,0)→sink(1,0)→sink(0,1)→ all neighbors exhausted
# (2,2)=1: count=2, sink(2,2)→ isolated
# → 2

def test_harness(fn):
    tests = [
        ([["1","1","1"],["0","1","0"],["1","1","1"]], 1),  # 1 big island
        ([["1","1","0"],["1","0","0"],["0","0","1"]], 2),  # 2 islands
        ([["1","1","0","0"],["1","1","0","0"],["0","0","1","0"],["0","0","0","1"]], 3),
        ([["0","0","0"],["0","0","0"]], 0),                 # all water
        ([["1"]], 1),                                       # single cell
    ]
    passed = 0
    for *inputs, expected in tests:
        grid_copy = copy.deepcopy(inputs[0])   # fn modifies grid in-place
        got = fn(grid_copy)
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness(num_islands)
print("num_islands defined.")

<a id='6'></a>
## 6. 🧩 Pattern 2: Clone Graph — LC 133

---

```
PROBLEM:
  Given a reference to a node in a connected undirected graph,
  return a deep copy (clone) of the graph.

TRICK:
  DFS + hash map (old_node → new_node).
  Before recursing into a neighbor, check the map:
  - If neighbor already cloned → use the cached clone (handles cycles).
  - If not → create a new clone, register it, then recurse.
  The map does double duty: visited set AND node registry.

SLOW MOTION TRACE on graph: 1—2—3—4—1 (cycle)

  clone(1): not in map → create clone_1, map[1]=clone_1
    clone(2): not in map → create clone_2, map[2]=clone_2
      clone(3): not in map → create clone_3, map[3]=clone_3
        clone(4): not in map → create clone_4, map[4]=clone_4
          clone(1): IN MAP → return map[1]=clone_1  ← cycle handled!
        clone_4.neighbors = [clone_3, clone_1]
      clone_3.neighbors = [clone_2, clone_4]
    clone_2.neighbors = [clone_1, clone_3]
  clone_1.neighbors = [clone_2, clone_4]

KEY INSIGHT:
  Register the clone in the map BEFORE recursing into neighbors.
  This prevents infinite loops in cyclic graphs.

TIME:  O(V+E) — visit each node and edge once
SPACE: O(V)   — hash map stores one clone per node
```

In [ ]:
class GraphNode:
    """LeetCode Node for Clone Graph problem."""
    def __init__(self, val=0, neighbors=None):
        self.val = val
        self.neighbors = neighbors if neighbors is not None else []

def clone_graph(node):
    """
    LC 133 — Clone Graph
    Approach: DFS with a hash map mapping original nodes to their clones.
    Args:
        node (GraphNode): any node in the connected undirected graph.
    Returns:
        GraphNode: the same node in the deep-copied graph.
    Time:  O(V+E) — each node and edge visited once
    Space: O(V)   — hash map stores all V clones
    """
    if not node:
        return None

    clones = {}   # maps original node → its clone; also acts as visited set

    def dfs(original):
        if original in clones:
            return clones[original]          # already cloned → return cached copy
        clone = GraphNode(original.val)      # create fresh clone with same value
        clones[original] = clone             # register BEFORE recursing (cycle guard)
        for neighbor in original.neighbors:
            clone.neighbors.append(dfs(neighbor))  # recursively clone each neighbor
        return clone

    return dfs(node)

# Slow motion on 1-2 (two nodes, mutual edges):
# dfs(1): not in clones → clone_1={val:1}, clones={1:clone_1}
#   dfs(2): not in clones → clone_2={val:2}, clones={1:clone_1, 2:clone_2}
#     dfs(1): IN clones → return clone_1
#   clone_2.neighbors = [clone_1]
#   return clone_2
# clone_1.neighbors = [clone_2]
# return clone_1

def test_harness(fn):
    # Build: 1—2—3—4, 1—4 (cycle)
    n1, n2, n3, n4 = GraphNode(1), GraphNode(2), GraphNode(3), GraphNode(4)
    n1.neighbors = [n2, n4]
    n2.neighbors = [n1, n3]
    n3.neighbors = [n2, n4]
    n4.neighbors = [n1, n3]

    cloned = fn(n1)

    # verify: same structure, different objects
    ok = (
        cloned is not n1 and
        cloned.val == 1 and
        cloned.neighbors[0].val == 2 and
        cloned.neighbors[1].val == 4 and
        cloned is not None
    )
    print("PASSED" if ok else "FAILED", "| clone is distinct:", cloned is not n1)
    print("cloned node val:", cloned.val, "neighbors:", [nb.val for nb in cloned.neighbors])

    # verify None input
    got_none = fn(None)
    print("PASSED" if got_none is None else "FAILED", "| None input → None output")
    print("2/2 tests passed" if ok and got_none is None else "some tests failed")

test_harness(clone_graph)
print("clone_graph defined.")

<a id='7'></a>
## 7. 🧩 Pattern 3: Pacific Atlantic Water Flow — LC 417

---

```
PROBLEM:
  Given a grid of heights, find all cells from which water can flow to BOTH
  the Pacific Ocean (top/left edges) AND the Atlantic Ocean (bottom/right edges).
  Water flows from higher or equal cells to lower or equal cells.

TRICK:
  Reverse the flow direction. Instead of water flowing DOWN from a cell,
  run DFS from the ocean edges going UPHILL (can reach if neighbor >= current).
  Pacific DFS: start from top row + left column.
  Atlantic DFS: start from bottom row + right column.
  Answer: cells reachable by BOTH DFS sets.

SLOW MOTION TRACE on 3x3 grid heights:
  [[1, 2, 2],
   [3, 2, 3],
   [2, 4, 5]]

  Pacific reachable (from top row + left col, going uphill):
    row 0 all: {(0,0),(0,1),(0,2)}
    col 0 all: {(0,0),(1,0),(2,0)}
    (1,0)=3 → can reach (1,1)=2? No (2<3). Can reach (2,0)=2? No.
    (0,1)=2 → (1,1)=2: yes (>=) → add (1,1) → (1,2)=3>=2 → add (1,2) → ...
    Pacific set includes: most of the grid from top-left

  Atlantic reachable (from bottom row + right col):
    (2,2)=5 → up to (1,2)=3, (2,1)=4 → propagates

  Intersection = cells in both sets.

KEY INSIGHT:
  Reverse-flow trick: DFS from the ocean going uphill avoids tracking multiple
  paths from each cell. Two DFS sweeps + intersection is elegant and O(R*C).

TIME:  O(R*C) — each cell visited at most twice (once per ocean DFS)
SPACE: O(R*C) — two visited sets
```

In [ ]:
def pacific_atlantic(heights):
    """
    LC 417 — Pacific Atlantic Water Flow
    Approach: Reverse DFS from each ocean boundary uphill; intersect the two reachable sets.
    Args:
        heights (List[List[int]]): grid of non-negative integers.
    Returns:
        List[List[int]]: list of [row, col] cells reachable by both oceans.
    Time:  O(R*C) — each cell visited at most twice
    Space: O(R*C) — two visited sets of up to R*C cells each
    """
    rows, cols = len(heights), len(heights[0])
    pacific  = set()   # cells that can drain to the Pacific (top/left)
    atlantic = set()   # cells that can drain to the Atlantic (bottom/right)

    def dfs(r, c, visited, prev_height):
        if (r, c) in visited or r < 0 or r >= rows or c < 0 or c >= cols:
            return
        if heights[r][c] < prev_height:
            return       # water can't flow uphill — prune this branch
        visited.add((r, c))
        for dr, dc in DIRS:
            dfs(r + dr, c + dc, visited, heights[r][c])  # go uphill from here

    # Pacific border: top row + left column
    for c in range(cols):
        dfs(0, c, pacific, heights[0][c])            # top row
    for r in range(rows):
        dfs(r, 0, pacific, heights[r][0])            # left column

    # Atlantic border: bottom row + right column
    for c in range(cols):
        dfs(rows-1, c, atlantic, heights[rows-1][c]) # bottom row
    for r in range(rows):
        dfs(r, cols-1, atlantic, heights[r][cols-1]) # right column

    # cells reachable from BOTH oceans
    return [[r, c] for r, c in pacific & atlantic]

# Slow motion sketch on heights=[[1,2,2],[3,2,3],[2,4,5]]:
# Pacific DFS from row0: reaches {(0,0),(0,1),(0,2)}
# Pacific DFS from col0: reaches (1,0)=3, (2,0)=2
# Atlantic DFS from row2: reaches (2,2)=5→(1,2)=3→(0,2)=2, (2,1)=4→...
# Intersection: cells reachable from both

def test_harness(fn):
    tests = [
        ([[1,2,2,3,5],[3,2,3,4,4],[2,4,5,3,1],[6,7,1,4,5],[5,1,1,2,4]],
         sorted([[0,4],[1,3],[1,4],[2,2],[3,0],[3,1],[4,0]])),
        ([[1]], [[0,0]]),    # single cell drains to both
        ([[1,1],[1,1]], sorted([[0,0],[0,1],[1,0],[1,1]])),
    ]
    passed = 0
    for *inputs, expected in tests:
        got = sorted(fn(inputs[0]))
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness(pacific_atlantic)
print("pacific_atlantic defined.")

<a id='8'></a>
## 8. 🧩 Pattern 4: Surrounded Regions — LC 130

---

```
PROBLEM:
  Given a board of 'X' and 'O', capture all regions of 'O' not connected
  to the board's border. Flip captured 'O's to 'X'.

TRICK:
  Any 'O' connected to the border CANNOT be captured.
  Step 1: DFS from every border 'O' → mark those as safe (use temp marker 'S').
  Step 2: Flip remaining 'O' → 'X' (they are surrounded).
  Step 3: Restore 'S' → 'O'.
  This is an "anti-BFS" — mark what NOT to flip, then flip everything else.

SLOW MOTION TRACE:
  Initial:         After step 1:    After step 2+3:
  X X X X          X X X X          X X X X
  X O O X          X S S X          X X X X   ← interior O captured
  X X O X    →     X X S X    →     X X X X
  X O X X          X O X X          X O X X   ← border O restored

  Border 'O' at (3,1) → safe → marked S
  Interior 'O' at (1,1),(1,2),(2,2) → surrounded → flipped to X

KEY INSIGHT:
  Mark safe cells from the border FIRST. Then flip everything unmarked.
  Always flip toward safety, not toward capture.

TIME:  O(R*C) — each cell visited at most once
SPACE: O(R*C) — recursion stack
```

In [ ]:
def solve(board):
    """
    LC 130 — Surrounded Regions
    Approach: DFS from border 'O's to mark safe; flip remaining 'O'→'X'; restore safe.
    Args:
        board (List[List[str]]): m x n grid of 'X' and 'O'. Modified in-place.
    Returns:
        None (modifies board in-place).
    Time:  O(R*C) — each cell touched at most a constant number of times
    Space: O(R*C) — recursion stack in worst case
    """
    if not board or not board[0]:
        return
    rows, cols = len(board), len(board[0])

    def mark_safe(r, c):
        if r < 0 or r >= rows or c < 0 or c >= cols or board[r][c] != 'O':
            return           # out of bounds or not an O — stop
        board[r][c] = 'S'    # temporarily mark as safe (connected to border)
        for dr, dc in DIRS:
            mark_safe(r + dr, c + dc)

    # step 1: DFS from all border 'O's → mark entire connected region as 'S'
    for r in range(rows):
        mark_safe(r, 0)          # left border
        mark_safe(r, cols - 1)   # right border
    for c in range(cols):
        mark_safe(0, c)          # top border
        mark_safe(rows - 1, c)   # bottom border

    # step 2 + 3: flip all 'O'→'X' (captured), restore all 'S'→'O' (safe)
    for r in range(rows):
        for c in range(cols):
            if board[r][c] == 'O':
                board[r][c] = 'X'    # surrounded — capture it
            elif board[r][c] == 'S':
                board[r][c] = 'O'    # safe — restore it

# Slow motion on:
# [X X X X]
# [X O O X]
# [X X O X]
# [X O X X]
# Border O: (3,1) → mark_safe(3,1)='S', neighbors all X → stop
# Interior O: (1,1),(1,2),(2,2) → remain 'O' → flipped to 'X'
# 'S' at (3,1) → restored to 'O'

def test_harness(fn):
    tests = [
        (
            [["X","X","X","X"],["X","O","O","X"],["X","X","O","X"],["X","O","X","X"]],
            [["X","X","X","X"],["X","X","X","X"],["X","X","X","X"],["X","O","X","X"]]
        ),
        ([["X"]], [["X"]]),
        ([["O"]], [["O"]]),  # single O on border — stays O
        (
            [["O","O"],["O","O"]],
            [["O","O"],["O","O"]]  # all border cells — none captured
        ),
    ]
    passed = 0
    for *inputs, expected in tests:
        board = copy.deepcopy(inputs[0])
        fn(board)
        got = board
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness(solve)
print("solve defined.")

<a id='9'></a>
## 9. 🧩 Pattern 5: Max Area of Island — LC 695

---

```
PROBLEM:
  Given a binary grid, find the maximum area of an island (connected 1s).

TRICK:
  Same structure as Number of Islands, but instead of counting islands,
  count the size of each island during DFS and track the maximum.
  DFS returns the size of the component it explored — accumulate with + 1 + sum(recursive calls).

SLOW MOTION TRACE on grid:
  [[0,0,1,0],
   [0,1,1,0],
   [0,1,0,0]]

  (0,2)=1: DFS:
    visit (0,2) → size=1
    go to (1,2): size=1
      go to (1,1): size=1
        go to (2,1): size=1
          no more unvisited neighbors → return 1
        return 1+1=2
      return 1+2=3
    return 1+3=4
  total island size = 4
  max_area = 4

KEY INSIGHT:
  DFS returns a size value. Summing return values accumulates the island area
  on the way back up the call stack.

TIME:  O(R*C) — each cell visited at most once
SPACE: O(R*C) — recursion depth in worst case
```

In [ ]:
def max_area_of_island(grid):
    """
    LC 695 — Max Area of Island
    Approach: DFS returns island size; track maximum across all islands.
    Args:
        grid (List[List[int]]): 2D binary grid (0 = water, 1 = land).
    Returns:
        int: area of the largest island (0 if none exists).
    Time:  O(R*C) — each cell visited at most once
    Space: O(R*C) — recursion stack
    """
    rows, cols = len(grid), len(grid[0])
    max_area = 0

    def dfs(r, c):
        if r < 0 or r >= rows or c < 0 or c >= cols or grid[r][c] != 1:
            return 0             # out of bounds or water or already visited
        grid[r][c] = 0           # mark visited by sinking (avoids extra visited set)
        # count this cell + area contributed by each neighbor
        return 1 + dfs(r+1,c) + dfs(r-1,c) + dfs(r,c+1) + dfs(r,c-1)

    for r in range(rows):
        for c in range(cols):
            if grid[r][c] == 1:        # start of an unvisited island
                area = dfs(r, c)       # get size of this island
                max_area = max(max_area, area)  # keep track of the biggest
    return max_area

# Slow motion on [[0,0,1,0],[0,1,1,0],[0,1,0,0]]:
# (0,2)=1: dfs(0,2)=1+dfs(1,2)
#          dfs(1,2)=1+dfs(1,1)
#          dfs(1,1)=1+dfs(2,1)
#          dfs(2,1)=1+0+0+0+0=1
#          chain returns: 1→2→3→4  max_area=4

def test_harness(fn):
    tests = [
        ([[0,0,1,0,0],[0,1,1,0,0],[0,1,0,0,0]], 4),   # island of size 4
        ([[0,0,0],[0,0,0]], 0),                         # all water
        ([[1,1,1],[1,1,1]], 6),                         # one big island
        ([[1,0,0,0],[0,0,0,0],[0,0,0,1]], 1),           # two isolated 1-cell islands
        ([[1,1,0,1,1],[0,1,0,1,0],[0,0,0,0,0]], 3),    # two islands of 3 and 3, max=3
    ]
    passed = 0
    for *inputs, expected in tests:
        grid_copy = copy.deepcopy(inputs[0])  # fn modifies grid in-place
        got = fn(grid_copy)
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness(max_area_of_island)
print("max_area_of_island defined.")

<a id='10'></a>
## 10. The Graphs DFS Decision Map

```
QUESTION TYPE                         KEY TECHNIQUE                   LC PROBLEMS
───────────────────────────────────────────────────────────────────────────────────
Count connected components            DFS outer loop, sink visited     200, 695
Max area / size of component          DFS returns accumulated size      695
Clone / deep copy a graph             DFS + old→new hash map           133
Reachable from boundaries             DFS from borders, mark safe      130, 417
Multi-source DFS                      DFS from multiple seeds          417
All paths source→destination          DFS + backtracking               797
Cycle detection (undirected)          DFS + visited + parent tracking  684
Cycle detection (directed)            DFS + gray/white/black coloring  207
Topological order                     DFS post-order reversed          207, 210
Flood fill                            DFS 4-directional, sink cells    733

DFS vs BFS:
  Use DFS: connected components, cycle detection, path existence, clone
  Use BFS: shortest path (unweighted), level-by-level, multi-source spread
```

<a id='11'></a>
## 11. Interview Cheat Sheet

**1. When to reach for Graph DFS:**

| Signal | What to Do |
|--------|------------|
| "connected components" | DFS outer loop, count launches |
| "flood fill" or "sink island" | DFS from each unvisited cell |
| "max area" of component | DFS returns size |
| "clone graph" | DFS + hash map (old→clone) |
| "reachable from border" | DFS inward from edges |
| "all paths" | DFS + backtracking |

**2. The O(V+E) operations — memorize these:**

```python
# GRID DFS: sink visited cells in-place
def dfs(r, c):
    if r < 0 or r >= R or c < 0 or c >= C or grid[r][c] != target:
        return
    grid[r][c] = VISITED
    for dr, dc in DIRS:
        dfs(r+dr, c+dc)

# GRAPH DFS: visited set
def dfs(node):
    visited.add(node)
    for neighbor in graph[node]:
        if neighbor not in visited:
            dfs(neighbor)

# CLONE: map + DFS
def dfs(node):
    if node in seen: return seen[node]
    clone = Node(node.val)
    seen[node] = clone
    for nb in node.neighbors:
        clone.neighbors.append(dfs(nb))
    return clone
```

**3. Common templates:**

```python
# TEMPLATE: COUNT CONNECTED COMPONENTS (grid)
count = 0
for r in range(rows):
    for c in range(cols):
        if grid[r][c] == 1:
            count += 1
            dfs(r, c)   # sinks entire component

# TEMPLATE: DFS THAT RETURNS SIZE
def dfs(r, c):
    if out_of_bounds or already_visited: return 0
    mark_visited
    return 1 + sum(dfs(r+dr, c+dc) for dr, dc in DIRS)

# TEMPLATE: MARK BORDER-CONNECTED, THEN FLIP
for r in range(rows):         # left + right border DFS
    dfs_safe(r, 0); dfs_safe(r, cols-1)
for c in range(cols):         # top + bottom border DFS
    dfs_safe(0, c); dfs_safe(rows-1, c)
# then flip 'O'→'X', 'S'→'O'

# TEMPLATE: REVERSE-FLOW DFS (LC 417 style)
def dfs(r, c, visited, prev_h):
    if (r,c) in visited or out_of_bounds: return
    if heights[r][c] < prev_h: return   # can't flow uphill
    visited.add((r,c))
    for dr, dc in DIRS:
        dfs(r+dr, c+dc, visited, heights[r][c])
```

**4. Gotchas to not forget:**

```
❌  Marking visited AFTER recursing — causes revisits and infinite loops in cycles
❌  Using DFS for shortest path — BFS guarantees shortest, DFS does NOT
❌  Forgetting to restore state when backtracking (all-paths problems)
❌  Not handling disconnected graphs — outer loop over all nodes required
✅  Sink cells in-place (0 or visited marker) to avoid a separate visited set
✅  Register clone node in map BEFORE recursing (handles cycles in clone graph)
✅  For border-mark problems: always DFS inward from ALL 4 edges, not just one
✅  DIRS is your friend — enumerate once at the top, use everywhere
```

## Summary Map

```
                    🔍 GRAPHS DFS
                         │
         ┌───────────────┼───────────────┐
         │               │               │
      GRID DFS      GRAPH DFS       BORDER DFS
         │               │               │
   ┌─────┴─────┐    ┌────┴────┐    ┌─────┴─────┐
   │           │    │         │    │           │
 COUNT      MAX     CLONE   CYCLE  MARK      REVERSE
 ISLANDS    AREA    GRAPH   DETECT SAFE      FLOW
 LC 200     LC 695  LC 133  LC 207 LC 130    LC 417

CORE DFS RULES:
  1. Mark visited BEFORE recursing
  2. Check bounds + visited at the TOP of DFS (not before calling)
  3. Outer loop handles disconnected graphs
  4. DFS = depth first, explores one path fully before trying others
  5. Backtracking = DFS + restore state after recursion
```

---
*End of Graphs DFS Master Guide — Sean Edition*